# EPFL Hackathon 2026 - Data Preprocesing

In [112]:
import numpy as np
import pandas as pd

from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import Aer
from qiskit_aer.primitives import Estimator 
from qiskit import transpile
from scipy.optimize import minimize

In [113]:
# 1) Load dataset
df = pd.read_csv("underwriting_dataset.csv")
df.columns = df.columns.str.strip()

# Use "v_i" directly (expected value of investigating)
v = df["v_i"].astype(float).to_numpy()
n = len(df)

# Choose exactly K to investigate (adjust as you want)
K = min(8, n)  # investigate 8 out of 24
lam = 1.0      # network synergy weight
rho = 200.0    # penalty weight for enforcing sum x_i = K

In [114]:
# 2) Build correlation/network matrix W (sparse, interpretable)
W = np.zeros((n, n), dtype=float)

def add_links(col, weight):
    vals = df[col].to_numpy()
    for i in range(n):
        for j in range(i + 1, n):
            if vals[i] == vals[j]:
                W[i, j] += weight
                W[j, i] += weight

add_links("garage_id",      1.0)
add_links("device_id",      1.0)
add_links("broker_id",      0.5)
add_links("ring_id_latent", 2.0)
add_links("region_id",      0.2)

# Optional: sparsify further by keeping only edges with weight >= threshold
threshold = 0.5
W[W < threshold] = 0.0

In [115]:
# 3) Build QUBO  

qp = QuadraticProgram("fraud_alert_prioritization")

for i in range(n):
    qp.binary_var(f"x_{i}")

linear = {}
quadratic = {}

def add_q(u, v, c):
    key = tuple(sorted((u, v)))
    quadratic[key] = quadratic.get(key, 0.0) + float(c)

# (A) -sum v_i x_i
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - float(v[i])

# (B) -lam * sum W_ij x_i x_j
for i in range(n):
    for j in range(i + 1, n):
        if W[i, j] != 0:
            add_q(f"x_{i}", f"x_{j}", -lam * float(W[i, j]))

# (C) EXACTLY-K constraint via penalty: rho*(sum_i x_i - K)^2
# Expand: rho*(sum x)^2 - 2*rho*K*(sum x) + const
# and (sum x)^2 = sum x_i + 2*sum_{i<j} x_i x_j because x_i^2 = x_i

for i in range(n):
    # from rho*(sum x)^2 -> +rho*x_i
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) + rho
    # from -2*rho*K*(sum x) -> -2*rho*K*x_i
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - 2.0 * rho * K

for i in range(n):
    for j in range(i + 1, n):
        # from rho*(sum x)^2 -> +2*rho*x_i*x_j
        add_q(f"x_{i}", f"x_{j}", 2.0 * rho)

qp.minimize(linear=linear, quadratic=quadratic)
qubo = QuadraticProgramToQubo().convert(qp)

In [116]:
# 4) Convert QUBO -> Ising
H, offset = qubo.to_ising()

reps = 2
ansatz = QAOAAnsatz(cost_operator=H, reps=reps)

estimator = Estimator()

def energy(theta):
    job = estimator.run([ansatz], [H], [theta])
    return float(job.result().values[0])

theta0 = np.random.default_rng(0).uniform(0, 2*np.pi, size=ansatz.num_parameters)
res_opt = minimize(energy, theta0, method="COBYLA", options={"maxiter": 200})
theta_star = res_opt.x

qc = ansatz.assign_parameters(theta_star)
qc.measure_all()

backend = Aer.get_backend("aer_simulator")
tqc = transpile(qc, backend)
counts = backend.run(tqc, shots=3000).result().get_counts()

best = max(counts, key=counts.get)

bits = np.array([int(b) for b in best[::-1]], dtype=int)

# Since the corrected QUBO has ONLY x_0..x_{n-1}, keep first n bits
x = bits[:n]
selected_idx = np.where(x == 1)[0]

In [118]:
# 5) Report 

value = float(np.sum(v * x))

synergy = 0.0
for i in range(n):
    for j in range(i + 1, n):
        synergy += W[i, j] * x[i] * x[j]
synergy = float(synergy)

print("Most frequent bitstring:", best)
print("Selected cases:", selected_idx.tolist())
print("Selected count:", int(x.sum()), "(target K =", K, ")")
print("Sum individual value:", value)
print("Network synergy term:", lam * synergy)
print("Total (value + synergy):", value + lam * synergy)

display(df.loc[selected_idx].reset_index(drop=True))

Most frequent bitstring: 101001000101000011111000
Selected cases: [3, 4, 5, 6, 7, 12, 14, 18, 21, 23]
Selected count: 10 (target K = 8 )
Sum individual value: 2474.7764008383
Network synergy term: 34.89999999999999
Total (value + synergy): 2509.6764008383


,case_id,claim_type,region_id,claim_amount,policy_age_days,prior_claims,broker_id,garage_id,device_id,review_minutes,p_fraud,ring_id_latent,severity_L,review_cost,v_i
0,3,auto,0,4160.428011,545,0,5,1,17,56,0.284220,1,1248.128403,56.0,139.108453
1,4,auto,4,1974.703798,1914,1,5,8,9,68,0.360451,1,592.411139,68.0,49.444431
2,5,auto,4,1864.767982,187,1,7,3,15,77,0.357810,3,559.430395,77.0,33.093401
3,6,health,3,3393.757134,1039,0,4,0,20,56,0.043008,0,678.751427,56.0,-39.944485
4,7,home,4,4920.212978,1618,0,3,3,25,53,0.273989,2,1968.085191,53.0,243.578013
5,12,auto,4,3759.536216,2598,0,0,3,25,52,0.290947,2,1127.860865,52.0,128.481259
6,14,auto,3,20918.088114,1168,1,4,8,4,74,0.561679,2,6275.426434,74.0,1864.625098
7,18,health,1,3021.150575,706,0,2,3,21,89,0.206564,3,604.230115,89.0,-20.353175
8,21,health,3,1092.420953,2667,0,4,3,1,66,0.138480,1,218.484191,66.0,-49.359406
9,23,auto,0,2957.815911,3610,1,7,1,2,58,0.377229,3,887.344773,58.0,126.102812
